# LeetCode 3731 — Find Missing Elements bằng Python

Link đề: https://leetcode.com/problems/find-missing-elements/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findMissingElements(self, nums: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `nums` gồm các số **đôi một khác nhau**, ban đầu chứa mọi số nguyên trong một đoạn `[min, max]` nhưng có thể thiếu vài số (min và max vẫn còn); hãy liệt kê các số bị thiếu theo thứ tự tăng.
2. Dùng `min`, `max`, `range` và `set` để tìm "phần tử thuộc đoạn nhưng không có trong mảng".
3. Hiểu hai góc nhìn: **duyệt từng giá trị trong đoạn** và **nhìn các khoảng trống giữa hai phần tử liền kề sau khi sắp xếp**.
4. Viết 5 cách: `in` trên list, `set`, sắp xếp + khoảng trống, mảng đánh dấu, và hiệu hai tập hợp; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: duyệt từ `1` thay vì `min`, quên sắp xếp, dùng `range(a, b)` thay vì `range(a + 1, b)`, và trả sai kiểu (số thay vì danh sách).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` gồm các phần tử **phân biệt**. Ban đầu mảng chứa **mọi** số nguyên trong một đoạn liên tục nào đó, nhưng một số bị mất; **số nhỏ nhất và số lớn nhất của đoạn vẫn còn trong `nums`**. Hãy trả về danh sách các số bị thiếu trong đoạn đó, theo thứ tự **tăng dần**; nếu không thiếu số nào thì trả về danh sách rỗng.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,4,2,5]` | `[3]` | Đoạn là `1..5`, chỉ thiếu `3` |
| 2 | `[7,8,6,9]` | `[]` | Đoạn là `6..9`, đủ cả |
| 3 | `[5,1]` | `[2,3,4]` | Đoạn là `1..5`, thiếu `2, 3, 4` |

Constraints: `2 <= nums.length <= 100`, `1 <= nums[i] <= 100`, các phần tử đôi một khác nhau.

In [1]:
for nums in [[1, 4, 2, 5], [7, 8, 6, 9], [5, 1]]:
    mn, mx = min(nums), max(nums)
    full = list(range(mn, mx + 1))
    missing = [x for x in full if x not in nums]
    print(f"nums={nums}  đoạn {mn}..{mx} = {full}  thiếu: {missing}")

nums=[1, 4, 2, 5]  đoạn 1..5 = [1, 2, 3, 4, 5]  thiếu: [3]
nums=[7, 8, 6, 9]  đoạn 6..9 = [6, 7, 8, 9]  thiếu: []
nums=[5, 1]  đoạn 1..5 = [1, 2, 3, 4, 5]  thiếu: [2, 3, 4]


## 3. Kiến thức nền

### 3.1. `min`, `max` và đoạn `[min, max]`

`min(lst)` và `max(lst)` trả về phần tử nhỏ/lớn nhất (`O(n)`). Đề cho biết `min` và `max` **chắc chắn còn trong `nums`**, nên đoạn ban đầu chính là `min(nums)..max(nums)`. Không cần đoán biên.

In [2]:
nums = [5, 1]
print(min(nums), max(nums), max(nums) - min(nums) + 1)      # số phần tử của đoạn đầy đủ

1 5 5


### 3.2. `range(a, b)` nửa mở

`range(a, b)` sinh `a, a+1, ..., b-1` (**không** gồm `b`). Muốn gồm cả `max` phải viết `range(mn, mx + 1)`. Muốn các số **nằm giữa** hai giá trị `a < b` (không gồm hai đầu) thì dùng `range(a + 1, b)`.

In [3]:
print(list(range(1, 6)), list(range(1, 5)))
print(list(range(1 + 1, 5)))          # chỉ các số nằm giữa 1 và 5
print(list(range(4, 5)), list(range(5, 5)))     # rỗng khi hai số liền kề

[1, 2, 3, 4, 5] [1, 2, 3, 4]
[2, 3, 4]
[4] []


### 3.3. List comprehension có điều kiện

`[x for x in iterable if cond]` xây danh sách mới từ các `x` thỏa `cond`, tương đương vòng `for` + `append` nhưng ngắn gọn.

In [4]:
nums = [1, 4, 2, 5]
present = set(nums)
print([x for x in range(1, 6) if x not in present])
res = []
for x in range(1, 6):
    if x not in present:
        res.append(x)
print(res)

[3]
[3]


### 3.4. `set` và phép trừ tập hợp

`set` kiểm tra `in` trong `O(1)`. Với hai tập `A`, `B`: `A - B` là các phần tử thuộc `A` mà không thuộc `B`. Vì `set` **không có thứ tự**, muốn kết quả tăng dần phải `sorted(...)`.

In [5]:
A = set(range(1, 6))
B = {1, 4, 2, 5}
print(A - B, sorted(A - B))
print(sorted({9, 3, 7} - {3}))

{3} [3]
[7, 9]


### 3.5. `sorted` và `zip` để xét cặp phần tử liền kề

`sorted(nums)` trả danh sách sắp xếp. `zip(a, a[1:])` ghép từng phần tử với phần tử kế tiếp: rất hợp để xét "khoảng trống" giữa hai số liền kề.

In [6]:
a = sorted([5, 1, 4, 2])
print(a)
print(list(zip(a, a[1:])))
for x, y in zip(a, a[1:]):
    print(x, y, "khoảng trống:", list(range(x + 1, y)))

[1, 2, 4, 5]
[(1, 2), (2, 4), (4, 5)]
1 2 khoảng trống: []
2 4 khoảng trống: [3]
4 5 khoảng trống: []


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Có hai cách nhìn cùng một bài toán:

- **Góc nhìn 1 (duyệt đoạn):** liệt kê mọi số từ `min` đến `max`, số nào không có trong `nums` thì ghi lại. Giống như điểm danh một lớp học theo số báo danh.
- **Góc nhìn 2 (khoảng trống):** sắp xếp `nums`; giữa hai số liền kề `x < y`, các số bị thiếu là `x+1, ..., y-1` (rỗng nếu `y = x + 1`).

Tính tay ví dụ 3: `nums = [5, 1]`. Đoạn `1..5` có `5` số, `nums` có `2` số, vậy thiếu `5 - 2 = 3` số. Sắp xếp: `[1, 5]`, khoảng trống giữa 1 và 5 là `2, 3, 4`.

**Đếm nhanh:** số phần tử thiếu luôn bằng `(max - min + 1) - len(nums)` (vì các phần tử phân biệt và đều nằm trong đoạn).

### Bảng tính tay

| nums | min..max | số phần tử đoạn | `len(nums)` | số thiếu | các số thiếu |
|---|---|---|---|---|---|
| `[1,4,2,5]` | 1..5 | 5 | 4 | 1 | 3 |
| `[7,8,6,9]` | 6..9 | 4 | 4 | 0 | (rỗng) |
| `[5,1]` | 1..5 | 5 | 2 | 3 | 2, 3, 4 |
| `[10,20]` | 10..20 | 11 | 2 | 9 | 11..19 |

In [7]:
for nums in [[1, 4, 2, 5], [7, 8, 6, 9], [5, 1], [10, 20]]:
    mn, mx = min(nums), max(nums)
    size = mx - mn + 1
    missing = sorted(set(range(mn, mx + 1)) - set(nums))
    print(nums, f"đoạn {mn}..{mx}", size, len(nums), "số thiếu =", size - len(nums), missing)
    assert len(missing) == size - len(nums)

[1, 4, 2, 5] đoạn 1..5 5 4 số thiếu = 1 [3]
[7, 8, 6, 9] đoạn 6..9 4 4 số thiếu = 0 []
[5, 1] đoạn 1..5 5 2 số thiếu = 3 [2, 3, 4]
[10, 20] đoạn 10..20 11 2 số thiếu = 9 [11, 12, 13, 14, 15, 16, 17, 18, 19]


## 5. Cách 1 — Brute Force / baseline

Với mỗi `x` trong `min..max`, kiểm tra `x in nums` **trực tiếp trên list**. Mỗi phép `in` quét `O(n)`, có `(max - min + 1)` giá trị cần kiểm tra.

Nút thắt: tích `O(n) * (max - min)` lặp lại phép quét giống nhau; chuyển `nums` sang cấu trúc tra cứu nhanh sẽ gọn hơn.

In [8]:
class SolutionBrute:
    def findMissingElements(self, nums):
        mn, mx = min(nums), max(nums)
        res = []
        for x in range(mn, mx + 1):
            if x not in nums:               # quét cả list
                res.append(x)
        return res

print(SolutionBrute().findMissingElements([1, 4, 2, 5]), SolutionBrute().findMissingElements([7, 8, 6, 9]), SolutionBrute().findMissingElements([5, 1]))

[3] [] [2, 3, 4]


**Độ phức tạp:** Time `O(n * R)` với `R = max - min + 1 <= 100`, Space `O(1)` ngoài kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách hai việc "ghi nhớ cái đã có" và "quét đoạn". Ghi nhớ bằng `set` (hoặc mảng đánh dấu), sau đó quét đoạn `min..max` đúng **một lần**.

**Quy tắc "nếu… thì…":**

- Nếu `x` thuộc `min..max` và **không** thuộc tập -> `x` bị thiếu.
- Vì quét theo thứ tự tăng nên kết quả **tự động tăng dần**, không cần sắp xếp lại.
- Nếu dùng sắp xếp, giữa hai số liền kề `x < y` -> thêm `x+1 .. y-1`.

**Invariant:** sau khi xử lý các giá trị `min..x`, kết quả chứa đúng các số bị thiếu trong `[min, x]`, theo thứ tự tăng.

In [9]:
# Kiểm chứng: số thiếu = (max - min + 1) - len(nums) với dữ liệu hợp lệ (phần tử phân biệt)
import random
random.seed(7)
for _ in range(2000):
    n = random.randint(2, 100)
    nums = random.sample(range(1, 101), n)
    mn, mx = min(nums), max(nums)
    missing = [x for x in range(mn, mx + 1) if x not in set(nums)]
    assert len(missing) == mx - mn + 1 - n
print("công thức đếm số phần tử thiếu đúng")

công thức đếm số phần tử thiếu đúng


## 7. Các cách tiếp theo

### Cách 2 — Dùng `set`

Tạo `present = set(nums)`; sau đó `[x for x in range(mn, mx + 1) if x not in present]`. Tra cứu `O(1)` nên tổng `O(n + R)`.

In [10]:
class SolutionSet:
    def findMissingElements(self, nums):
        present = set(nums)
        mn, mx = min(nums), max(nums)
        return [x for x in range(mn, mx + 1) if x not in present]

print(SolutionSet().findMissingElements([1, 4, 2, 5]), SolutionSet().findMissingElements([7, 8, 6, 9]), SolutionSet().findMissingElements([5, 1]))

[3] [] [2, 3, 4]


**Độ phức tạp:** Time `O(n + R)`, Space `O(n)`.

### Cách 3 — Sắp xếp rồi xét khoảng trống

Sắp xếp `nums`; với mỗi cặp liền kề `(x, y)` thêm `range(x + 1, y)` vào kết quả. Cách này **không phụ thuộc cận giá trị** (hợp cả khi `nums[i]` rất lớn) và kết quả đã tăng dần.

In [11]:
class SolutionSorted:
    def findMissingElements(self, nums):
        a = sorted(nums)
        res = []
        for x, y in zip(a, a[1:]):
            res.extend(range(x + 1, y))     # các số nằm giữa x và y
        return res

print(SolutionSorted().findMissingElements([1, 4, 2, 5]), SolutionSorted().findMissingElements([7, 8, 6, 9]), SolutionSorted().findMissingElements([5, 1]))

[3] [] [2, 3, 4]


**Độ phức tạp:** Time `O(n log n + k)` với `k` là số phần tử thiếu, Space `O(n)`.

### Cách 4 — Mảng đánh dấu

Vì `nums[i] <= 100`, dùng `seen = [False] * 101`, đánh dấu từng `v`, rồi quét `mn..mx`. Không cần băm, bộ nhớ hằng số.

In [12]:
class SolutionArray:
    def findMissingElements(self, nums):
        seen = [False] * 101
        for v in nums:
            seen[v] = True
        mn, mx = min(nums), max(nums)
        res = []
        for x in range(mn, mx + 1):
            if not seen[x]:
                res.append(x)
        return res

print(SolutionArray().findMissingElements([1, 4, 2, 5]), SolutionArray().findMissingElements([7, 8, 6, 9]), SolutionArray().findMissingElements([5, 1]))

[3] [] [2, 3, 4]


**Độ phức tạp:** Time `O(n + R)`, Space `O(1)` (mảng 101 ô cố định).

### Cách 5 — Hiệu hai tập hợp

Tập đầy đủ `set(range(mn, mx + 1))` trừ `set(nums)`, rồi `sorted`. Ngắn nhất, đọc gần như câu đề.

In [13]:
class SolutionSetDiff:
    def findMissingElements(self, nums):
        return sorted(set(range(min(nums), max(nums) + 1)) - set(nums))

print(SolutionSetDiff().findMissingElements([1, 4, 2, 5]), SolutionSetDiff().findMissingElements([7, 8, 6, 9]), SolutionSetDiff().findMissingElements([5, 1]))

[3] [] [2, 3, 4]


**Độ phức tạp:** Time `O(n + R + k log k)`, Space `O(n + R)`.

## 8. TRACE từng bước

Trace Cách 3 (khoảng trống) với `nums = [5, 1, 4]`: sau sắp xếp `[1, 4, 5]`.

| cặp (x, y) | `range(x+1, y)` | kết quả tích lũy |
|---|---|---|
| (1, 4) | 2, 3 | `[2, 3]` |
| (4, 5) | (rỗng) | `[2, 3]` |

Trace Cách 2 (set) với `nums = [1, 4, 2, 5]`: `present = {1, 2, 4, 5}`; `x = 1, 2` có, `x = 3` thiếu, `x = 4, 5` có -> `[3]`.

In [14]:
def trace_gaps(nums):
    a = sorted(nums)
    print("sau sắp xếp:", a)
    res = []
    for x, y in zip(a, a[1:]):
        gap = list(range(x + 1, y))
        res.extend(gap)
        print(f"  cặp ({x},{y}): khoảng trống {gap} -> kết quả {res}")
    return res

assert trace_gaps([5, 1, 4]) == [2, 3]
assert trace_gaps([7, 8, 6, 9]) == []

sau sắp xếp: [1, 4, 5]
  cặp (1,4): khoảng trống [2, 3] -> kết quả [2, 3]
  cặp (4,5): khoảng trống [] -> kết quả [2, 3]
sau sắp xếp: [6, 7, 8, 9]
  cặp (6,7): khoảng trống [] -> kết quả []
  cặp (7,8): khoảng trống [] -> kết quả []
  cặp (8,9): khoảng trống [] -> kết quả []


In [15]:
def trace_set(nums):
    present = set(nums)
    mn, mx = min(nums), max(nums)
    print(f"present={sorted(present)} đoạn {mn}..{mx}")
    res = []
    for x in range(mn, mx + 1):
        if x in present:
            print(f"  x={x}: có mặt")
        else:
            print(f"  x={x}: THIẾU -> thêm")
            res.append(x)
    print("  kết quả:", res)
    return res

assert trace_set([1, 4, 2, 5]) == [3]

present=[1, 2, 4, 5] đoạn 1..5
  x=1: có mặt
  x=2: có mặt
  x=3: THIẾU -> thêm
  x=4: có mặt
  x=5: có mặt
  kết quả: [3]


## 9. Code nộp LeetCode

In [16]:
from typing import List, Optional

class Solution:
    def findMissingElements(self, nums: List[int]) -> List[int]:
        present = set(nums)
        return [x for x in range(min(nums), max(nums) + 1) if x not in present]

## 10. Vì sao thuật toán đúng

- **Tập ứng viên đúng:** đề đảm bảo đoạn ban đầu có min và max nằm trong `nums`, nên đoạn đó là `[min(nums), max(nums)]`. Bị thiếu nghĩa là thuộc đoạn nhưng không thuộc `nums`.
- **Đủ và không thừa:** ta xét mọi `x` trong đoạn đúng một lần và chỉ giữ `x not in present`, nên kết quả chứa **đúng** các số thiếu, không trùng lặp.
- **Thứ tự tăng:** `range` sinh các giá trị tăng dần và comprehension giữ nguyên thứ tự.
- **Cách khoảng trống:** sau khi sắp xếp, các số thiếu nằm xen giữa hai phần tử liền kề; ngoài `[min, max]` không có số nào cần xét, và hai số liền kề `y = x + 1` cho khoảng trống rỗng.
- **Kiểm chứng số lượng:** do các phần tử phân biệt, số thiếu `= (max - min + 1) - len(nums)`; test ở mục 12 kiểm tra đẳng thức này.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| `in` trên list | `O(n * R)` | `O(1)` |
| `set` | `O(n + R)` | `O(n)` |
| Sắp xếp + khoảng trống | `O(n log n + k)` | `O(n)` |
| Mảng đánh dấu | `O(n + R)` | `O(1)` |
| Hiệu hai tập hợp | `O(n + R + k log k)` | `O(n + R)` |

`n = len(nums)`, `R = max - min + 1`, `k` = số phần tử thiếu. Trong bài này `R <= 100` và `n <= 100` nên mọi cách đều tức thì; cách sắp xếp là cách duy nhất vẫn tốt khi giá trị rất lớn mà `n` nhỏ (nếu chỉ cần đếm hoặc liệt kê các khoảng).

## 12. Bộ test

Ví dụ của đề, các biên (`n = 2`, mảng liền kề, mảng đủ cả đoạn, `1` và `100`) và so sánh **tất cả cách** trên nhiều input hợp lệ (các phần tử phân biệt).

In [17]:
impls = [SolutionBrute(), SolutionSet(), SolutionSorted(), SolutionArray(), SolutionSetDiff(), Solution()]

cases = [
    ([1, 4, 2, 5], [3]), ([7, 8, 6, 9], []), ([5, 1], [2, 3, 4]),
    ([1, 2], []), ([2, 1], []), ([1, 100], list(range(2, 100))),
    ([100, 1, 50], [x for x in range(2, 100) if x != 50]),
    (list(range(1, 101)), []), ([3, 3 + 2], [4]), ([10, 12, 14], [11, 13]),
]
for nums, exp in cases:
    assert 2 <= len(nums) <= 100 and len(set(nums)) == len(nums) and all(1 <= v <= 100 for v in nums)
    for impl in impls:
        got = impl.findMissingElements(list(nums))
        assert got == exp, (type(impl).__name__, nums, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 10 ca


In [18]:
import random
random.seed(3731)
for _ in range(800):
    n = random.randint(2, 100)
    nums = random.sample(range(1, 101), n)
    if random.random() < 0.3:                          # mảng "gần như đầy đủ"
        lo = random.randint(1, 50)
        hi = random.randint(lo + 1, 100)
        pool = list(range(lo, hi + 1))
        keep = [x for x in pool if x in (lo, hi) or random.random() < 0.8]
        random.shuffle(keep)
        nums = keep
    exp = [x for x in range(min(nums), max(nums) + 1) if x not in set(nums)]
    outs = [impl.findMissingElements(list(nums)) for impl in impls]
    assert all(o == exp for o in outs), (nums, outs, exp)
    assert exp == sorted(exp) and len(exp) == max(nums) - min(nums) + 1 - len(nums)
    assert set(exp).isdisjoint(nums) and all(min(nums) < x < max(nums) for x in exp)
print("800 test ngẫu nhiên (seed=3731): 6 cách giống nhau và thỏa các tính chất")

800 test ngẫu nhiên (seed=3731): 6 cách giống nhau và thỏa các tính chất


In [19]:
# Hợp của nums và kết quả phải là cả đoạn min..max
for _ in range(300):
    nums = random.sample(range(1, 101), random.randint(2, 100))
    r = Solution().findMissingElements(nums)
    assert sorted(nums + r) == list(range(min(nums), max(nums) + 1))
print("nums ∪ missing = [min, max] (không trùng) OK")

nums ∪ missing = [min, max] (không trùng) OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: duyệt từ `1` thay vì từ `min(nums)`

Đề chỉ xét đoạn `[min, max]`; các số nhỏ hơn `min` **không** bị coi là thiếu.

In [20]:
def wrong_from_one(nums):
    present = set(nums)
    return [x for x in range(1, max(nums) + 1) if x not in present]       # SAI: bắt đầu từ 1

print(wrong_from_one([7, 8, 6, 9]), "!=", Solution().findMissingElements([7, 8, 6, 9]))
assert wrong_from_one([7, 8, 6, 9]) == [1, 2, 3, 4, 5] != Solution().findMissingElements([7, 8, 6, 9])

[1, 2, 3, 4, 5] != []


### Lỗi 2: dùng `range(x, y)` thay vì `range(x + 1, y)`

`range(x, y)` đưa cả `x` (đang có mặt) vào kết quả.

In [21]:
def wrong_gap_includes_left(nums):
    a = sorted(nums)
    res = []
    for x, y in zip(a, a[1:]):
        res.extend(range(x, y))                    # SAI: gồm x
    return res

print(wrong_gap_includes_left([1, 4, 2, 5]), "!=", Solution().findMissingElements([1, 4, 2, 5]))
assert wrong_gap_includes_left([1, 4, 2, 5]) == [1, 2, 3, 4] != [3]

[1, 2, 3, 4] != [3]


### Lỗi 3: quên sắp xếp trước khi xét cặp liền kề

Cặp liền kề **trong mảng gốc** không phải cặp liền kề về giá trị.

In [22]:
def wrong_no_sort(nums):
    res = []
    for x, y in zip(nums, nums[1:]):
        res.extend(range(x + 1, y))                # SAI: nums chưa sắp xếp
    return res

print(wrong_no_sort([5, 1]), "!=", Solution().findMissingElements([5, 1]))
assert wrong_no_sort([5, 1]) == [] != Solution().findMissingElements([5, 1])

[] != [2, 3, 4]


### Lỗi 4: trả về số lượng hoặc phần tử đầu thay vì danh sách

Đề yêu cầu **danh sách** mọi số thiếu, không phải số đầu tiên hay số lượng.

In [23]:
def wrong_first_only(nums):
    present = set(nums)
    for x in range(min(nums), max(nums) + 1):
        if x not in present:
            return [x]                             # SAI: dừng ở số thiếu đầu tiên
    return []

print(wrong_first_only([5, 1]), "!=", Solution().findMissingElements([5, 1]))
assert wrong_first_only([5, 1]) == [2] != [2, 3, 4]

[2] != [2, 3, 4]


### Lỗi 5: sắp xếp tại chỗ làm thay đổi đầu vào

`nums.sort()` thay đổi mảng người gọi truyền vào; nên dùng `sorted(nums)` nếu không cần ghi đè.

In [24]:
def risky_sort_in_place(nums):
    nums.sort()                                    # có tác dụng phụ lên đầu vào
    return [y for x, y in zip(nums, nums[1:])]

data = [5, 1, 4]
risky_sort_in_place(data)
print("đầu vào sau khi gọi:", data)
data2 = [5, 1, 4]
Solution().findMissingElements(data2)
assert data != [5, 1, 4] and data2 == [5, 1, 4]

đầu vào sau khi gọi: [1, 4, 5]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm các giá trị còn thiếu trong một đoạn" -> `set` + quét đoạn, hoặc sắp xếp + nhìn khoảng trống.
- Đề đảm bảo `min`, `max` còn -> không cần lo biên; dùng `min(nums)` và `max(nums)` trực tiếp.
- Hiệu tập hợp `set(range(...)) - set(nums)` cho lời giải một dòng (nhớ `sorted`).
- Số phần tử thiếu có công thức `(max - min + 1) - n` nếu các phần tử phân biệt -> dùng để tự kiểm tra.
- `range(a + 1, b)` cho các số nằm **giữa** hai giá trị.
- Với giá trị rất lớn nhưng ít phần tử, dùng sắp xếp + khoảng trống (không duyệt cả đoạn).

## 15. Biến thể của bài

1. **Missing Number** (LeetCode 268): thiếu đúng một số trong `0..n`; dùng tổng `n(n+1)/2` hoặc XOR.
2. **Find All Numbers Disappeared in an Array** (448): tìm các số thiếu trong `1..n` khi có trùng lặp.
3. **Summary Ranges** (228): gom các số liền nhau thành khoảng.
4. **Missing Ranges** (163, Premium): liệt kê các khoảng thiếu trong `[lower, upper]`.
5. **First Missing Positive** (41): yêu cầu `O(n)` thời gian và `O(1)` bộ nhớ phụ.
6. **Find Missing and Repeated Values** (2965): một số lặp, một số thiếu trong ma trận.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `in` trên list | quét list mỗi lần | `O(n * R)` | `O(1)` | Oracle kiểm tra |
| `set` | tra cứu `O(1)` rồi quét đoạn | `O(n + R)` | `O(n)` | Nộp bài, rõ ràng |
| Sắp xếp + khoảng trống | `range(x + 1, y)` giữa cặp liền kề | `O(n log n + k)` | `O(n)` | Giá trị rất lớn, ít phần tử |
| Mảng đánh dấu | `seen[v]` cỡ 101 | `O(n + R)` | `O(1)` | Giá trị có cận nhỏ |
| Hiệu hai tập | `set(range) - set(nums)` | `O(n + R + k log k)` | `O(n + R)` | Viết nhanh một dòng |

Nộp cách `set`.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().findMissingElements([10, 14, 11, 13])` ra gì?

**Bài 2 (tính tay):** với `nums = [3, 9, 5]`, đoạn là gì, thiếu bao nhiêu số (dùng công thức), liệt kê chúng.

**Bài 3 (điền TODO):** hoàn thành `count_missing(nums)` trả về **số lượng** số thiếu mà không tạo danh sách.

**Bài 4 (tự viết lại):** viết `find_missing_ranges(nums)` trả về các khoảng thiếu dạng `(lo, hi)` thay vì từng số, ví dụ `[3, 9, 5] -> [(4, 4), (6, 8)]`.

**Bài 5 (mini challenge):** viết `find_missing_with_bounds(nums, lo, hi)` cho phiên bản **không** đảm bảo min/max còn: liệt kê mọi số trong `[lo, hi]` không có trong `nums`.

In [25]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().findMissingElements([10, 14, 11, 13]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_missing_todo(nums):
    # TODO: dùng công thức (max - min + 1) - len(nums)
    return 0

[12]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: đoạn `10..14`, bạn đã thấy `10, 11, 13, 14`. Bài 2: min `3`, max `9`.

**Mức 2 (vừa):** Bài 3: các phần tử phân biệt nên đếm được bằng một phép trừ. Bài 4: sau khi sắp xếp, khoảng trống giữa `(x, y)` là `(x + 1, y - 1)` nếu `y - x > 1`. Bài 5: duyệt `range(lo, hi + 1)`.

**Mức 3 (gần đáp án):** Bài 1: `[12]`. Bài 2: đoạn `3..9` có `7` số, `nums` có `3` -> thiếu `4` số: `[4, 6, 7, 8]`. Bài 4: `[(4, 4), (6, 8)]`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().findMissingElements([10, 14, 11, 13]) == [12]
assert Solution().findMissingElements([3, 9, 5]) == [4, 6, 7, 8]
assert 9 - 3 + 1 - 3 == 4

# Bài 3
def count_missing(nums):
    return max(nums) - min(nums) + 1 - len(nums)
assert count_missing([3, 9, 5]) == 4 and count_missing([7, 8, 6, 9]) == 0 and count_missing([5, 1]) == 3
assert all(count_missing(x) == len(Solution().findMissingElements(x)) for x in ([1, 4, 2, 5], [1, 100], [2, 3, 7]))

# Bài 4
def find_missing_ranges(nums):
    a = sorted(nums)
    out = []
    for x, y in zip(a, a[1:]):
        if y - x > 1:
            out.append((x + 1, y - 1))
    return out
assert find_missing_ranges([3, 9, 5]) == [(4, 4), (6, 8)]
assert find_missing_ranges([7, 8, 6, 9]) == []
assert find_missing_ranges([5, 1]) == [(2, 4)]

# Bài 5
def find_missing_with_bounds(nums, lo, hi):
    present = set(nums)
    return [x for x in range(lo, hi + 1) if x not in present]
assert find_missing_with_bounds([3, 9, 5], 1, 10) == [1, 2, 4, 6, 7, 8, 10]
assert find_missing_with_bounds([3, 9, 5], 3, 9) == Solution().findMissingElements([3, 9, 5])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tìm phần tử thiếu trong đoạn":**

```python
present = set(nums)
return [x for x in range(min(nums), max(nums) + 1) if x not in present]
# hoặc: sắp xếp rồi với mỗi cặp liền kề (x, y) thêm range(x + 1, y)
```

**Checklist trước khi submit:**

- [ ] Đoạn là `[min(nums), max(nums)]`, không bắt đầu từ `1`?
- [ ] `range(mn, mx + 1)` (gồm `mx`) hoặc `range(x + 1, y)` (không gồm hai đầu)?
- [ ] Kết quả **tăng dần** và là **danh sách** (rỗng `[]` khi không thiếu)?
- [ ] Không làm thay đổi đầu vào (`sorted` thay vì `sort`)?
- [ ] Thử `[1, 2]` (không thiếu), `[1, 100]` (thiếu 98 số), `[5, 1]`?
- [ ] Chữ ký đúng `findMissingElements(self, nums: List[int]) -> List[int]`?